# 15. 항목별 최악 이상치 날짜 시간별 분석
각 측정항목에서 물리적/통계적 기준으로 가장 심한 이상치를 기록한 계량기의 해당 날짜 시간별 데이터를 확인합니다.

In [1]:
import sys
from pathlib import Path

ROOT = Path('/home/aceya/EMS')
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
from ems.db import load_env, connect

load_env()

# (meter, measurement, target_date, 이상치 설명)
TARGETS = [
    # 전류 I1
    ('H1.Z15', 'I1',    '2018-01-01', 'I1 물리적이상 최악: min=-446.6A'),
    ('H1.Z17', 'I1',    '2022-07-19', 'I1 통계적이상 최악: max=429.1A'),
    # 전류 I2
    ('H1.Z15', 'I2',    '2018-01-01', 'I2 물리적이상 최악: min=-444.2A'),
    ('H1.Z17', 'I2',    '2022-07-19', 'I2 통계적이상 최악: max=422.6A'),
    # 전류 I3
    ('H1.Z15', 'I3',    '2018-01-01', 'I3 물리적이상 최악: max=-11.6A (전기간 음수)'),
    ('H1.Z17', 'I3',    '2022-07-19', 'I3 통계적이상 최악: max=416.6A'),
    # 전력 P
    ('H1.Z15', 'P',     '2018-01-01', 'P 물리적이상 최악: min=-137383W'),
    ('H1.Z17', 'P',     '2022-07-20', 'P 통계적이상 최악: max=251694W'),
    # 전력 P1
    ('H1.Z15', 'P1',    '2018-01-01', 'P1 물리적이상 최악: min=-45087W'),
    ('H1.Z17', 'P1',    '2022-07-20', 'P1 통계적이상 최악: max=86093W'),
    # 전력 P2
    ('H1.Z15', 'P2',    '2018-01-01', 'P2 물리적이상 최악: min=-46390W'),
    ('H1.Z17', 'P2',    '2022-07-20', 'P2 통계적이상 최악: max=85295W'),
    # 전력 P3
    ('H1.Z15', 'P3',    '2018-01-01', 'P3 물리적이상 최악: min=-48164W'),
    ('H2.ZE67','P3',    '2023-12-21', 'P3 통계적이상 최악: max=248380W'),
    # 역률 PF
    ('H1.ZE20','PF',    '2023-01-02', 'PF 물리적이상 최악: min=-1.0003, max=1.0109'),
    ('H3.Z40', 'PF',    '2020-01-24', 'PF 통계적이상 최악: min=0.0'),
    # 역률 PF1
    ('H3.Z312','PF1',   '2021-06-15', 'PF1 물리적이상 최악: min=-1.0'),
    ('H2.ZE66','PF1',   '2022-03-22', 'PF1 통계적이상 최악: max=1.0'),
    # 역률 PF2
    ('H2.Z311','PF2',   '2021-07-18', 'PF2 물리적이상 최악: min=-1.0'),
    ('H2.ZE66','PF2',   '2022-03-22', 'PF2 통계적이상 최악: max=1.0'),
    # 역률 PF3
    ('H1.Z310','PF3',   '2021-06-08', 'PF3 물리적이상 최악: min=-1.0'),
    ('H1.Z17', 'PF3',   '2022-07-03', 'PF3 통계적이상 최악: max=1.0'),
    # 전압 U1
    ('H2.T.Z34','U1',   '2020-03-07', 'U1 물리적이상 최악: min=0.0V'),
    ('H3.Z312', 'U1',   '2023-07-16', 'U1 통계적이상 최악: max=238.8V'),
    # 전압 U2
    ('H2.T.Z34','U2',   '2020-03-07', 'U2 물리적이상 최악: min=0.0V'),
    ('H3.Z312', 'U2',   '2022-03-26', 'U2 통계적이상 최악: max=239.4V'),
    # 전압 U3
    ('H2.T.Z34','U3',   '2020-03-07', 'U3 물리적이상 최악: min=0.0V'),
    ('H3.Z312', 'U3',   '2023-01-07', 'U3 통계적이상 최악: max=237.9V'),
    # 주파수 f
    ('H2.T.Z34','f',    '2020-03-07', 'f 물리적이상 최악: min=0.0Hz'),
    ('H2.Z65',  'f',    '2018-03-01', 'f 통계적이상 최악: max=50.08Hz'),
    # 누적에너지 WQ
    ('H2.Z64',  'WQ',   '2018-01-01', 'WQ 물리적이상 최악: min=-303021 (전기간 음수)'),
    # 누적에너지 WQ_out
    ('H1.Z25',  'WQ_out','2018-01-24','WQ_out 물리적이상 최악: min=-76.4'),
    ('H1.Z23',  'WQ_out','2023-12-22','WQ_out 통계적이상 최악: max=1851'),
    # 누적에너지 W_out
    ('H2.Z65',  'W_out','2022-05-20', 'W_out 물리적이상 최악: min=-0.048'),
    ('H1.Z28',  'W_out','2018-01-17', 'W_out 통계적이상 최악: max=117463048'),
    # 누적에너지 W_in
    ('H1.Z17',  'W_in', '2018-01-01', 'W_in 통계적이상 최악: max=591144922'),
    # 누적에너지 W
    ('H1.Z19',  'W',    '2018-03-02', 'W 물리적이상 최악: min=-161.2'),
    # 누적에너지 WQ_in
    ('H2.T.Z30','WQ_in','2018-01-03', 'WQ_in 통계적이상 최악: max=2288'),
    # 누적에너지 W3
    ('H2.ZE65', 'W3',   '2022-03-22', 'W3 물리적이상 최악: min=-3081.3'),
    ('V.ZE84',  'W3',   '2022-11-23', 'W3 통계적이상 최악: max=10358'),
    # 누적에너지 W2
    ('H2.ZE74', 'W2',   '2022-03-18', 'W2 물리적이상 최악: min=-1.90'),
    # 무효전력 Q
    ('H2.T.Z34','Q',    '2018-01-30', 'Q 물리적이상 최악: min=-24509VAR'),
    ('H2.T.Z33','Q',    '2019-08-27', 'Q 통계적이상 최악: max=87165VAR'),
    # 열량계 Tdiff
    ('H1.K15',  'Tdiff','2018-01-22', 'Tdiff 물리적이상 최악(냉각): min=-18963mK'),
    ('H1.W11',  'Tdiff','2023-09-21', 'Tdiff 통계적이상 최악(난방): max=38713mK'),
    # 열량계 Trl
    ('H1.K15',  'Trl',  '2018-02-23', 'Trl 물리적이상 최악: min=-151.8°C'),
    ('H1.W11',  'Trl',  '2023-08-01', 'Trl 통계적이상 최악: max=63.2°C'),
    # 열량계 Tvl
    ('H1.K15',  'Tvl',  '2018-02-23', 'Tvl 물리적이상 최악: min=-43.8°C'),
    ('H1.W11',  'Tvl',  '2023-08-01', 'Tvl 통계적이상 최악: max=78.4°C'),
    # 열량계 qv
    ('H1.K15',  'qv',   '2018-08-31', 'qv 물리적이상 최악: min=-0 (부동소수점)'),
    ('H1.K15',  'qv',   '2018-01-03', 'qv 통계적이상 최악: max=122.6 m3/h'),
]

print(f'총 {len(TARGETS)}개 대상')

총 51개 대상


In [2]:
def fetch_hourly(meter_urn, measurement, date_str):
    sql = """
        SELECT
            (ts AT TIME ZONE 'Europe/Berlin') AS ts_local,
            value
        FROM ems.cr_measurement_1h
        WHERE meter_urn = %s
          AND measurement = %s
          AND DATE(ts AT TIME ZONE 'Europe/Berlin') = %s
        ORDER BY ts
    """
    with connect() as conn:
        df = pd.read_sql(sql, conn, params=(meter_urn, measurement, date_str))
    return df


all_results = []

for meter, meas, date, desc in TARGETS:
    df = fetch_hourly(meter, meas, date)
    if df.empty:
        print(f'[데이터없음] {meter} {meas} {date}')
        continue
    df['meter'] = meter
    df['measurement'] = meas
    df['description'] = desc
    all_results.append(df)
    print(f'\n=== {desc} ===')
    print(f'{meter} {meas} {date}: {len(df)}행, min={df["value"].min():.4f}, max={df["value"].max():.4f}')
    print(df[['ts_local', 'value']].to_string(index=False))

/tmp/ipykernel_125541/1478685746.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, date_str))
/tmp/ipykernel_125541/1478685746.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, date_str))



=== I1 물리적이상 최악: min=-446.6A ===
H1.Z15 I1 2018-01-01: 24행, min=-132.7513, max=-51.1814
           ts_local       value
2018-01-01 00:00:00 -100.971556
2018-01-01 01:00:00  -78.761072
2018-01-01 02:00:00 -129.687107
2018-01-01 03:00:00  -99.460040
2018-01-01 04:00:00 -113.092102
2018-01-01 05:00:00 -130.737200
2018-01-01 06:00:00 -131.749414
2018-01-01 07:00:00 -130.651056
2018-01-01 08:00:00 -131.392025
2018-01-01 09:00:00 -132.751293
2018-01-01 10:00:00 -131.788793
2018-01-01 11:00:00 -129.062361
2018-01-01 12:00:00 -109.469038
2018-01-01 13:00:00 -131.958963
2018-01-01 14:00:00 -130.795508
2018-01-01 15:00:00 -131.656825
2018-01-01 16:00:00 -131.771861
2018-01-01 17:00:00 -129.656301
2018-01-01 18:00:00 -128.737898
2018-01-01 19:00:00 -128.003820
2018-01-01 20:00:00 -128.789438
2018-01-01 21:00:00  -79.127876
2018-01-01 22:00:00  -64.688857
2018-01-01 23:00:00  -51.181356

=== I1 통계적이상 최악: max=429.1A ===
H1.Z17 I1 2022-07-19: 24행, min=179.6014, max=426.5048
           ts_local     

/tmp/ipykernel_125541/1478685746.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, date_str))
/tmp/ipykernel_125541/1478685746.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, date_str))



=== I2 물리적이상 최악: min=-444.2A ===
H1.Z15 I2 2018-01-01: 24행, min=93.8248, max=137.0035
           ts_local      value
2018-01-01 00:00:00 118.316407
2018-01-01 01:00:00 107.195974
2018-01-01 02:00:00 132.749717
2018-01-01 03:00:00 117.238506
2018-01-01 04:00:00 124.394326
2018-01-01 05:00:00 133.974427
2018-01-01 06:00:00 134.034705
2018-01-01 07:00:00 133.549278
2018-01-01 08:00:00 129.940327
2018-01-01 09:00:00 133.718655
2018-01-01 10:00:00 135.295979
2018-01-01 11:00:00 134.919445
2018-01-01 12:00:00 121.775484
2018-01-01 13:00:00 134.785394
2018-01-01 14:00:00 136.790770
2018-01-01 15:00:00 135.942710
2018-01-01 16:00:00 135.323979
2018-01-01 17:00:00 135.642721
2018-01-01 18:00:00 137.003545
2018-01-01 19:00:00 134.655580
2018-01-01 20:00:00 134.671513
2018-01-01 21:00:00 108.410738
2018-01-01 22:00:00 100.478894
2018-01-01 23:00:00  93.824787

=== I2 통계적이상 최악: max=422.6A ===
H1.Z17 I2 2022-07-19: 24행, min=181.0907, max=422.6312
           ts_local      value
2022-07-19 00:00:00 

/tmp/ipykernel_125541/1478685746.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, date_str))
/tmp/ipykernel_125541/1478685746.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, date_str))


...

/tmp/ipykernel_125541/1478685746.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, date_str))
/tmp/ipykernel_125541/1478685746.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, date_str))



=== Trl 통계적이상 최악: max=63.2°C ===
H1.W11 Trl 2023-08-01: 24행, min=33.9333, max=61.7417
           ts_local     value
2023-08-01 00:00:00 40.000000
2023-08-01 01:00:00 39.750000
2023-08-01 02:00:00 39.000000
2023-08-01 03:00:00 38.433333
2023-08-01 04:00:00 38.000000
2023-08-01 05:00:00 37.041667
2023-08-01 06:00:00 37.438889
2023-08-01 07:00:00 55.811111
2023-08-01 08:00:00 54.641667
2023-08-01 09:00:00 51.575000
2023-08-01 10:00:00 48.866667
2023-08-01 11:00:00 48.000000
2023-08-01 12:00:00 47.108333
2023-08-01 13:00:00 45.733333
2023-08-01 14:00:00 44.941667
2023-08-01 15:00:00 40.775000
2023-08-01 16:00:00 37.858333
2023-08-01 17:00:00 36.916667
2023-08-01 18:00:00 35.833333
2023-08-01 19:00:00 34.850000
2023-08-01 20:00:00 33.933333
2023-08-01 21:00:00 47.033333
2023-08-01 22:00:00 61.741667
2023-08-01 23:00:00 58.866667

=== Tvl 물리적이상 최악: min=-43.8°C ===
H1.K15 Tvl 2018-02-23: 24행, min=-43.7583, max=22.0250
           ts_local      value
2018-02-23 00:00:00  10.308333
2018-02-23 0

/tmp/ipykernel_125541/1478685746.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, date_str))
/tmp/ipykernel_125541/1478685746.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, date_str))



=== Tvl 통계적이상 최악: max=78.4°C ===
H1.W11 Tvl 2023-08-01: 24행, min=33.6000, max=62.0333
           ts_local     value
2023-08-01 00:00:00 40.000000
2023-08-01 01:00:00 39.241667
2023-08-01 02:00:00 38.933333
2023-08-01 03:00:00 38.000000
2023-08-01 04:00:00 37.416667
2023-08-01 05:00:00 36.983333
2023-08-01 06:00:00 37.733333
2023-08-01 07:00:00 58.683333
2023-08-01 08:00:00 59.202778
2023-08-01 09:00:00 52.297222
2023-08-01 10:00:00 49.141667
2023-08-01 11:00:00 47.850000
2023-08-01 12:00:00 46.941667
2023-08-01 13:00:00 45.641667
2023-08-01 14:00:00 44.441667
2023-08-01 15:00:00 40.700000
2023-08-01 16:00:00 37.175000
2023-08-01 17:00:00 36.000000
2023-08-01 18:00:00 35.441667
2023-08-01 19:00:00 34.441667
2023-08-01 20:00:00 33.600000
2023-08-01 21:00:00 50.783333
2023-08-01 22:00:00 62.033333
2023-08-01 23:00:00 58.858333

=== qv 물리적이상 최악: min=-0 (부동소수점) ===
H1.K15 qv 2018-08-31: 24행, min=-0.0000, max=4.4653
           ts_local         value
2018-08-31 00:00:00  1.071917e+00
2018-08

/tmp/ipykernel_125541/1478685746.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, date_str))


In [3]:
if all_results:
    save_dir = ROOT / 'outputs/tables/anomaly'
    save_dir.mkdir(parents=True, exist_ok=True)
    final = pd.concat(all_results, ignore_index=True)
    final.to_csv(save_dir / 'anomaly_hourly.csv', index=False)
    print(f'총 {len(final)}행 저장 완료')

총 1126행 저장 완료
